In [ ]:
# Install dependencies into the current notebook environment
%pip -q install sahi pycocotools opencv-python ultralytics


In [25]:
import json

# Replace with your actual file path
json_file = "plastic_coco/annotation/test.json"

with open(json_file, 'r') as f:
    data = json.load(f)

print("--- TOP LEVEL KEYS ---")
print(list(data.keys()))

print("\n--- SAMPLE IMAGE ENTRY (First Item) ---")
if 'images' in data and len(data['images']) > 0:
    print(json.dumps(data['images'][0], indent=4))
else:
    print("No images found or key is missing.")

print("\n--- SAMPLE ANNOTATION ENTRY (First Item) ---")
if 'annotations' in data and len(data['annotations']) > 0:
    print(json.dumps(data['annotations'][0], indent=4))
else:
    print("No annotations found or key is missing.")

print("\n--- SAMPLE CATEGORY ENTRY (First Item) ---")
if 'categories' in data and len(data['categories']) > 0:
    print(json.dumps(data['categories'][0], indent=4))
else:
    print("No categories found.")

--- TOP LEVEL KEYS ---
['info', 'licenses', 'images', 'annotations', 'categories']

--- SAMPLE IMAGE ENTRY (First Item) ---
{
    "license": 1,
    "file_name": "000001.png",
    "coco_url": "",
    "height": 480,
    "width": 640,
    "date_captured": "",
    "flickr_url": "",
    "id": 1074
}

--- SAMPLE ANNOTATION ENTRY (First Item) ---
{
    "segmentation": {
        "size": [
            480,
            640
        ],
        "counts": "f^]33l>100000O\\Vl5"
    },
    "area": 18,
    "iscrowd": 1,
    "image_id": 1074,
    "bbox": [
        233.0,
        245.0,
        5.0,
        4.0
    ],
    "category_id": 1,
    "id": 38700
}

--- SAMPLE CATEGORY ENTRY (First Item) ---
{
    "supercategory": "plastic_litter",
    "id": 1,
    "name": "plastic_litter"
}


## Old

## JSON Cleanup + Slicing + COCO to YOLO Conversion

### 1. JSON Cleanup

Involves:
- getting rid of nonstandard keys
- converting RLE to polygon
- getting rid of very small polygons that cause noise & slice failure

In [ ]:
import json
import os
import shutil
import numpy as np
from shapely.geometry import Polygon, MultiPolygon
from shapely.validation import make_valid
from pycocotools import mask as mask_utils
import cv2

# ================= CONFIGURATION =================
RAW_DATA_ROOT = "plastic_coco"
TEMP_JSON_DIR = "plastic_coco/temp_fixed_json"
SLICED_DATA_DIR = "plastic_sliced_coco"
YOLO_OUTPUT_DIR = "plastic_sliced_yolo"

SLICE_SIZE = 640 
OVERLAP_RATIO = 0.2
MIN_AREA = 5.0 # If polygon area < 5px, delete segmentation

DATA_SPLITS = [
    ("train", "train.json", "images/train"), 
    ("val",   "val.json",   "images/val"),
    ("test",  "test.json",  "images/test")
]
# =================================================

def rle_to_polygon(segmentation, height, width):
    """ Converts RLE to Polygon (OpenCV Style). """
    try:
        if isinstance(segmentation, dict) and isinstance(segmentation.get('counts'), str):
            segmentation['counts'] = segmentation['counts'].encode('utf-8')
            binary_mask = mask_utils.decode([segmentation])[:, :, 0]
        elif isinstance(segmentation, dict) and isinstance(segmentation.get('counts'), list):
            rle = mask_utils.frPyObjects(segmentation, height, width)
            binary_mask = mask_utils.decode(rle)
        elif isinstance(segmentation, list):
            rle = mask_utils.frPyObjects(segmentation, height, width)
            binary_mask = mask_utils.decode(rle)
        else:
            return []

        contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        polygons = []
        for contour in contours:
            if contour.size >= 6: 
                polygons.append(contour.flatten().tolist())
        return polygons
    except Exception:
        return []

def fix_polygon_geometry(poly):
    """
    Magic fix for broken polygons:
    1. make_valid() handles topological errors.
    2. buffer(0) unties self-intersections.
    """
    if not poly.is_valid:
        poly = make_valid(poly)
    
    # buffer(0) is the "nuclear option" for fixing bowties
    poly = poly.buffer(0)
    
    # If fixing created a MultiPolygon (island), take the largest piece
    if isinstance(poly, MultiPolygon):
        if len(poly.geoms) > 0:
            poly = max(poly.geoms, key=lambda p: p.area)
        else:
            return None
            
    if poly.is_empty or poly.area < MIN_AREA:
        return None
        
    return poly

def scorched_earth_cleaning(annotations):
    clean_anns = []
    fixed_coords = 0
    removed_segs = 0
    
    for ann in annotations:
        # 1. Sanitize BBox (Backup Safety)
        if 'bbox' in ann:
            x, y, w, h = ann['bbox']
            if w < 2: w = 2
            if h < 2: h = 2
            ann['bbox'] = [x, y, w, h]
            bbox_area = w * h
        else:
            continue

        # 2. Fix Segmentation
        valid_seg = False
        
        if 'segmentation' in ann and isinstance(ann['segmentation'], list) and len(ann['segmentation']) > 0:
            try:
                # Grab the main polygon
                raw_coords = ann['segmentation'][0]
                if len(raw_coords) >= 6:
                    pts = np.array(raw_coords).reshape(-1, 2)
                    shapely_poly = Polygon(pts)
                    
                    # --- THE FIX ---
                    # Clean the geometry
                    clean_poly = fix_polygon_geometry(shapely_poly)
                    
                    if clean_poly is not None:
                        # CRITICAL: Overwrite coordinates in JSON
                        # Convert back to flat list: [x1, y1, x2, y2...]
                        new_coords = np.array(clean_poly.exterior.coords).flatten().tolist()
                        ann['segmentation'] = [new_coords]
                        ann['area'] = clean_poly.area
                        ann['iscrowd'] = 0
                        valid_seg = True
                        fixed_coords += 1
            except Exception:
                valid_seg = False

        # 3. Final Decision
        if not valid_seg:
            # Delete broken segmentation, fallback to BBox
            if 'segmentation' in ann:
                del ann['segmentation']
            
            ann['area'] = bbox_area
            ann['iscrowd'] = 0
            removed_segs += 1
        
        clean_anns.append(ann)

    print(f"    - Scorched Earth Report:")
    print(f"      FIXED: {fixed_coords} polygons (geometry repaired & coordinates updated)")
    print(f"      REMOVED: {removed_segs} segmentations (fallback to BBox)")
    return clean_anns

def fix_json_phase1(split_name, json_filename):
    """ Phase 1: RLE -> Scorched Earth Fix -> Save """
    input_json = os.path.join(RAW_DATA_ROOT, "annotation", json_filename)
    output_json = os.path.join(TEMP_JSON_DIR, json_filename)
    
    print(f"Fixing JSON for {split_name.upper()}...")
    with open(input_json, 'r') as f:
        data = json.load(f)

    img_dims = {img['id']: (img['height'], img['width']) for img in data['images']}
    processed_anns = []

    # Step A: RLE Conversion
    for ann in data['annotations']:
        if ann['image_id'] not in img_dims: continue
        
        if 'segmentation' in ann and isinstance(ann['segmentation'], dict) and 'counts' in ann['segmentation']:
            polygons = rle_to_polygon(ann['segmentation'], *img_dims[ann['image_id']])
            if polygons:
                ann['segmentation'] = polygons
                processed_anns.append(ann)
        else:
            processed_anns.append(ann)

    # Step B: Scorched Earth Cleaning
    data['annotations'] = scorched_earth_cleaning(processed_anns)
    
    os.makedirs(os.path.dirname(output_json), exist_ok=True)
    with open(output_json, 'w') as f:
        json.dump(data, f)
    
    return output_json

# ================= EXECUTION =================

if os.path.exists(YOLO_OUTPUT_DIR): shutil.rmtree(YOLO_OUTPUT_DIR)
if os.path.exists(TEMP_JSON_DIR): shutil.rmtree(TEMP_JSON_DIR)
if os.path.exists(SLICED_DATA_DIR): shutil.rmtree(SLICED_DATA_DIR)


# --- PHASE 1 ---
for split, json_file, src_folder in DATA_SPLITS:
    fixed_json_path = fix_json_phase1(split, json_file)


Fixing JSON for TRAIN...
    - Scorched Earth Report:
      FIXED: 72271 polygons (geometry repaired & coordinates updated)
      REMOVED: 1450 segmentations (fallback to BBox)
Fixing JSON for VAL...
    - Scorched Earth Report:
      FIXED: 22524 polygons (geometry repaired & coordinates updated)
      REMOVED: 436 segmentations (fallback to BBox)
Fixing JSON for TEST...
    - Scorched Earth Report:
      FIXED: 22057 polygons (geometry repaired & coordinates updated)
      REMOVED: 211 segmentations (fallback to BBox)


### 2. Slicing

Use sahi to slice image to uniform size

In [24]:
from sahi.slicing import slice_coco
import os

def slice_dataset(split_name, json_path, src_folder_name):
    """ Phase 2: Slice """
    image_dir = os.path.join(RAW_DATA_ROOT, src_folder_name)
    output_dir = os.path.join(SLICED_DATA_DIR, 'images', split_name)
    
    print(f"Slicing {split_name.upper()}...")
    
    coco_dict, result_path = slice_coco(
        coco_annotation_file_path=json_path,
        image_dir=image_dir,
        output_coco_annotation_file_name=split_name,
        output_dir=output_dir,
        slice_height=SLICE_SIZE,
        slice_width=SLICE_SIZE,
        overlap_height_ratio=OVERLAP_RATIO,
        overlap_width_ratio=OVERLAP_RATIO,
        min_area_ratio=0.1, 
        verbose=False 
    )

    return result_path, output_dir


sliced_json_paths = [] 

for split, json_file, src_folder in DATA_SPLITS:
    fixed_json_path = os.path.join(TEMP_JSON_DIR, json_file)
    sliced_json_path, sliced_img_dir = slice_dataset(split, fixed_json_path, src_folder)
    sliced_json_paths.append((split, sliced_json_path, sliced_img_dir))


Slicing TRAIN...


100%|██████████| 2226/2226 [02:36<00:00, 14.26it/s]


Slicing VAL...


100%|██████████| 742/742 [01:03<00:00, 11.72it/s]


Slicing TEST...


100%|██████████| 741/741 [01:00<00:00, 12.31it/s]


In [25]:
from pathlib import Path
import shutil

for split, json_path, _ in sliced_json_paths:
    new_name = json_path.stem.replace("_coco", "") + json_path.suffix
    destination = Path(f"{SLICED_DATA_DIR}/annotated") / new_name

    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(json_path, destination)

    print(f"Moved {json_path} -> {destination}")


Moved plastic_sliced_coco/images/train/train_coco.json -> plastic_sliced_coco/annotated/train.json
Moved plastic_sliced_coco/images/val/val_coco.json -> plastic_sliced_coco/annotated/val.json
Moved plastic_sliced_coco/images/test/test_coco.json -> plastic_sliced_coco/annotated/test.json


### 3. Convert Sliced COCO JSON to YOLO txt

In [ ]:
# --- PHASE 3 ---

from ultralytics.data.converter import convert_coco

print("\n--- PHASE 3: Generating YOLO Labels ---")

convert_coco(
    labels_dir=os.path.join(SLICED_DATA_DIR, "annotated"),
    save_dir=YOLO_OUTPUT_DIR,
    use_segments=True,
    use_keypoints=False,
)



--- PHASE 3: Generating YOLO Labels ---
Annotations /home/user/projects/ai-plastic-spotter/train/plastic_sliced_coco/annotated/test.json: 100% ━━━━━━━━━━━━ 2126/2126 1.7Kit/s 1.2s0.0s
Annotations /home/user/projects/ai-plastic-spotter/train/plastic_sliced_coco/annotated/train.json: 100% ━━━━━━━━━━━━ 6215/6215 1.6Kit/s 3.9s0.0s
Annotations /home/user/projects/ai-plastic-spotter/train/plastic_sliced_coco/annotated/val.json: 100% ━━━━━━━━━━━━ 2019/2019 1.8Kit/s 1.1s0.0s
COCO data converted successfully.
Results saved to /home/user/projects/ai-plastic-spotter/train/plastic_sliced_yolo


### 4. Symlink Original Sliced Image to YOLO Folder

Symlink: plastic_sliced_yolo/images/* -> plastic_sliced_coco/images/*

For compressing this with tar use the `-h` flag:


```sh
tar -h -czvf plastic_sliced_yolo.tar.gz plastic_sliced_yolo/
```

> tar `-h` flag: 
> *When reading or writing a file to be archived, tar accesses the file that a symbolic link points to, rather than the symlink itself.*

In [27]:

# 4. Link Images

def safe_symlink(split_name, src_subfolder):
    """ 
    Creates a relative symlink.
    Checks if source exists FIRST to prevent broken 'file-like' links.
    """
    # 1. Define paths
    # The actual physical location of the images
    target_abs_path = os.path.abspath(src_subfolder)
    
    # Where we want the link to be
    link_parent = os.path.join(YOLO_OUTPUT_DIR, "images")
    link_path = os.path.join(link_parent, split_name)

    # 2. VALIDATION: Does the source actually exist?
    if not os.path.exists(target_abs_path):
        print(f"  [CRITICAL ERROR] Cannot link '{split_name}'. Source folder not found:")
        print(f"    -> {target_abs_path}")
        print(f"    -> Check 'DATA_SPLITS' config at top of script.")
        return

    # 3. Cleanup existing links
    os.makedirs(link_parent, exist_ok=True)
    if os.path.lexists(link_path): # lexists returns True even for broken links
        if os.path.islink(link_path) or os.path.isfile(link_path):
            os.unlink(link_path)
        else:
            shutil.rmtree(link_path)

    # 4. Create RELATIVE Symlink (More robust in WSL)
    # This creates a link pointing to e.g. "../../../plastic_coco/images/train"
    rel_source = os.path.relpath(target_abs_path, start=link_parent)
    
    print(f"Linking: {link_path} -> {rel_source}")
    try:
        os.symlink(rel_source, link_path)
    except OSError as e:
        print(f"  [Error] Failed to create symlink: {e}")


print("\n--- PHASE 4: Linking Image Folders ---")
for split, _, src_folder in sliced_json_paths:
    safe_symlink(split, src_folder)

print("\nDONE.")


--- PHASE 4: Linking Image Folders ---
Linking: plastic_sliced_yolo/images/train -> ../../plastic_sliced_coco/images/train
Linking: plastic_sliced_yolo/images/val -> ../../plastic_sliced_coco/images/val
Linking: plastic_sliced_yolo/images/test -> ../../plastic_sliced_coco/images/test

DONE.


## Inspecting Anomalous Polygons

In [28]:
import json

# Path to your RAW train.json
json_path = "plastic_coco/temp_fixed_json/train.json"

with open(json_path, 'r') as f:
    data = json.load(f)

# 1. Get the 187th Image (Index 186)
# Note: SAHI iterates images in the order they appear in 'images' list
if len(data['images']) > 186:
    target_img = data['images'][186]
    print(f"--- SUSPECT IMAGE (Index 186) ---")
    print(f"ID: {target_img['id']}")
    print(f"File: {target_img['file_name']}")
    print(f"Dims: {target_img['width']}x{target_img['height']}")
    
    # 2. Find annotations for this image
    anns = [a for a in data['annotations'] if a['image_id'] == target_img['id']]
    print(f"Found {len(anns)} annotations.")
    
    # 3. Inspect them
    for i, ann in enumerate(anns):
        print(ann)
        print(f"\n[Annotation {i}] ID: {ann['id']}")
        print(f"  - BBox: {ann.get('bbox')}")
        print(f"  - Area (in JSON): {ann.get('area')}")
        print(f"  - Iscrowd: {ann.get('iscrowd')}")
        
        # Check Segmentation
        seg = ann.get('segmentation')
        if not seg:
            print("  - Segmentation: NONE/EMPTY")
        elif isinstance(seg, dict):
             print(f"  - Segmentation: RLE (Counts len: {len(seg.get('counts', ''))})")
        elif isinstance(seg, list):
             # Check if it looks like a line
             points = len(seg[0]) if len(seg) > 0 else 0
             print(f"  - Segmentation: POLYGON (Points: {points})")
             if points < 6:
                 print("    [WARNING] Polygon has < 3 points (Invalid!)")
else:
    print("Dataset has fewer than 186 images.")

--- SUSPECT IMAGE (Index 186) ---
ID: 3696
File: 001163.png
Dims: 1600x1200
Found 19 annotations.
{'segmentation': [[1553.0, 648.0, 1551.0, 650.0, 1551.0, 651.0, 1548.0, 654.0, 1548.0, 655.0, 1547.0, 656.0, 1547.0, 660.0, 1546.0, 661.0, 1546.0, 674.0, 1561.0, 689.0, 1563.0, 689.0, 1564.0, 690.0, 1565.0, 690.0, 1566.0, 691.0, 1567.0, 691.0, 1568.0, 692.0, 1569.0, 692.0, 1570.0, 693.0, 1572.0, 693.0, 1573.0, 694.0, 1574.0, 694.0, 1575.0, 695.0, 1576.0, 695.0, 1577.0, 696.0, 1579.0, 696.0, 1580.0, 697.0, 1581.0, 697.0, 1582.0, 698.0, 1583.0, 698.0, 1584.0, 699.0, 1586.0, 699.0, 1587.0, 700.0, 1588.0, 700.0, 1589.0, 701.0, 1590.0, 701.0, 1592.0, 703.0, 1593.0, 703.0, 1594.0, 704.0, 1595.0, 704.0, 1597.0, 706.0, 1598.0, 706.0, 1599.0, 707.0, 1599.0, 652.0, 1595.0, 652.0, 1594.0, 653.0, 1587.0, 653.0, 1586.0, 652.0, 1583.0, 652.0, 1582.0, 651.0, 1578.0, 651.0, 1577.0, 650.0, 1572.0, 650.0, 1571.0, 649.0, 1566.0, 649.0, 1565.0, 648.0, 1553.0, 648.0]], 'area': 2225.0, 'iscrowd': 0, 'image_id':

## Train Prep

### Intel (XPU)

In [29]:
import torch

print(f"PyTorch Version: {torch.__version__}")

print(f"XPU Available: {torch.xpu.is_available()}")

if torch.xpu.is_available():
    print(f"Current GPU: {torch.xpu.get_device_name(0)}")

device_str = 'xpu' if torch.xpu.is_available() else 'cpu'
device = torch.device('xpu' if torch.xpu.is_available() else 'cpu')

# Ultralytics has hardcoded CUDA calls that break on XPU
# Monkey-patch functions to work with XPU
import torch
from ultralytics import YOLO
import ultralytics.engine.trainer as trainer_module
import ultralytics.engine.validator as validator_module
import ultralytics.utils.torch_utils as torch_utils_module
import ultralytics.utils.checks as checks_module
import ultralytics.utils as utils_module
import gc

def _get_memory_xpu(self, fraction=False):
    """XPU-compatible memory getter."""
    memory = torch.xpu.memory_reserved() if hasattr(torch.xpu, 'memory_reserved') else 0
    if fraction:
        return 0.1  # Return low fraction to skip memory clearing
    return memory / 2**30

def _clear_memory_xpu(self, threshold=0.5):
    """XPU-compatible memory clearing."""
    gc.collect()
    if hasattr(torch.xpu, 'empty_cache'):
        torch.xpu.empty_cache()

# Patch the select_device function to handle XPU
_original_select_device = torch_utils_module.select_device

def select_device_xpu(device="", newline=False, verbose=True):
    """XPU-aware device selector."""
    device_str = str(device).strip().lower()
    if device_str == "xpu" or (hasattr(device, 'type') and device.type == 'xpu'):
        if torch.xpu.is_available():
            return torch.device('xpu')
    return _original_select_device(device, newline, verbose)

# Patch check_amp to avoid CUDA-only probing on non-CUDA builds (XPU/CPU)
_original_check_amp = checks_module.check_amp

def check_amp_safe(model):
    try:
        if hasattr(torch, "xpu") and torch.xpu.is_available():
            return False
        if not torch.cuda.is_available():
            return False
    except Exception:
        return False
    return _original_check_amp(model)

checks_module.check_amp = check_amp_safe
# BaseTrainer imports check_amp into its module namespace, so patch there too
trainer_module.check_amp = check_amp_safe

# Apply all patches
trainer_module.BaseTrainer._get_memory = _get_memory_xpu
trainer_module.BaseTrainer._clear_memory = _clear_memory_xpu

# Patch select_device everywhere it's imported
torch_utils_module.select_device = select_device_xpu
trainer_module.select_device = select_device_xpu
validator_module.select_device = select_device_xpu

# Also patch in ultralytics.utils (where it might be imported from)
utils_module.torch_utils.select_device = select_device_xpu

PyTorch Version: 2.9.1+xpu
XPU Available: True
Current GPU: Intel(R) Graphics [0x64a0]


### NVIDIA (CUDA)

In [ ]:
from ultralytics import YOLO

## Train

In [30]:
# "-seg" is crucial! It tells YOLO to learn masks, not just boxes.
model = YOLO("yolo11n-seg.pt") 

# intel moment.
# Ultralytics uses AMP auto-checks that hard-require CUDA; disable AMP on XPU/CPU builds
amp_flag = False if device_str in {'xpu', 'cpu'} else True

# 2. Train the model
results = model.train(
    data="bepliv1_seg.yml", # Path to your config file
    epochs=1,               # Number of training rounds
    imgsz=640,                # Image size (640 is standard)
    batch=16,                 # Reduce this if you run out of GPU memory
    # device=0,                 # Use '0' for GPU, 'cpu' for CPU
    project="plastic_project",# Folder name for saving results
    name="11n-seg",            # Sub-folder name
    
    # Segmentation specific settings (optional but good to know)
    overlap_mask=True,        # Masks should overlap during training
    mask_ratio=4,             # Downsample ratio for masks (4 is standard)

    # intel moment.
    device=device_str,
    amp=amp_flag,
)

# 3. Validation
print("Training Complete. Validating...")
metrics = model.val()
print(f"mAP50-95 (Box): {metrics.box.map}")
print(f"mAP50-95 (Mask): {metrics.seg.map}")


New https://pypi.org/project/ultralytics/8.3.245 available 😃 Update with 'pip install -U ultralytics'
engine/trainer: agnostic_nms=False, amp=False, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=bepliv1_seg.yml, degrees=0.0, deterministic=True, device=xpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=11n-seg6, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=

KeyboardInterrupt: 

## Test

In [ ]:
from pathlib import Path
from PIL import Image
from ultralytics import YOLO
from IPython.display import display

best_weights = Path('plastic_project/train/weights/best.pt')

if not best_weights.exists():
    raise FileNotFoundError(f"{best_weights} not found; rerun training or adjust the path.")

imgs = ['beach1.jpg', 'beach2.jpg', 'beach3.jpg']

infer_model = YOLO(str(best_weights))

metrics = infer_model.val(split='test') 
print(metrics)

for img in imgs:
    test_image = Path(img)
    if not test_image.exists():
        raise FileNotFoundError(f"{test_image} not found; place the sample image alongside the notebook.")
    
    results = infer_model(test_image, conf=0.5)
    for result in results:
        im_bgr = result.plot()  # includes masks if present
        if result.masks is not None:
            print('masks:', result.masks.data.shape)
        else:
            print('masks: None')
        display(Image.fromarray(im_bgr))
